In [62]:
import re
import os
import json
import openai
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS

In [63]:
# Load the API key into the client for OpenAI API access.
def load_api_key(file_path='api_key.txt'):
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('api_key='):
                return line.strip().split('=', 1)[1]
    return None

os.environ["OPENAI_API_KEY"] = load_api_key()

api_key = load_api_key()

if api_key is None:
    print("Error: API key not found.")
    exit()

client = OpenAI(api_key=api_key,)

chat_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Say this is a test",
        }
    ]
)

In [64]:
class LangChainActions:
    def __init__(self, env):
        #self.chat_openai = ChatOpenAI(model="gpt-4o-mini").with_structured_output(output)
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        with open('tools/actions.json') as actions_json:
            self.actions = json.load(actions_json)

        with open('tools/rooms.json') as rooms_json:
            self.rooms = json.load(rooms_json)

        with open('tools/objects.json') as objects_json:
            self.objects = json.load(objects_json)

        with open('tools/graph_config.json') as config_json:
            self.config = json.load(config_json)
        
        self.object_index = FAISS.load_local("tools/objects_index", self.embeddings, allow_dangerous_deserialization=True)
        self.room_index = FAISS.load_local("tools/room_index", self.embeddings, allow_dangerous_deserialization=True)
        self.action_index = FAISS.load_local("tools/actions_index", self.embeddings, allow_dangerous_deserialization=True)

        self.object_index.delete(self.objects.keys())

        # all object's that's in env_# will have at least one room in "rooms"
        # resulting metadata:
        # [
        #     {"rooms": ["livingroom", "bedroom"], "keyword": "rug"},
        #     {"rooms": ["bathroom", "bedroom"], "keyword": "ceilinglamp"}
        # ]
        metadatas = []
        for key in self.objects.keys():
            room_metadata = {"rooms": [], "keyword": self.objects[key]['keyword']}
            for x in self.config[f'env_{env}']:
                if x['class_name'] == self.objects[key]['keyword']:
                    for room in x['rooms']:
                        room_metadata['rooms'].append(room['class_name'])
            metadatas.append(room_metadata)


        self.object_index.add_texts(
            self.objects.keys(),
            ids=self.objects.keys(),
            metadatas=metadatas
        )
    
    def get_action(self, action):
        action_key = self.action_index.similarity_search_with_relevance_scores(action, k=1)
        if action_key[0][1] >= 0.8:
            return action_key[0][0].page_content
        # action_key = [
        #     (ActionPage(page_content="[sit]"), 0.95)  # A tuple of the match and its relevance score
        # ]
        return None

    def get_room(self, room):
        room_key = self.room_index.similarity_search_with_relevance_scores(room, k=1)
        if room_key[0][1] >= 0.9:
            return room_key[0][0].page_content
        return None
    
    # based on the object in the certain env to do embeddings.
    def get_object(self, object, filter=None):
        object_key = self.object_index.similarity_search_with_relevance_scores(object, k=1, filter=filter)
        if len(object_key) == 0:
            return None
        # object_key = [
        #     (ObjectPage(metadata={"keyword": "rug"}), 0.85)
        # ]
        if object_key[0][1] >= 0.6:
            return object_key[0][0].metadata['keyword']
        return None

In [65]:
# Load prompts from the file
def load_prompts_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()  # Read the entire file content

    # Split the content into system and user sections using markers
    system_prompt = content.split('--USER--')[0].replace('--SYSTEM--', '').strip()
    user_prompt = content.split('--USER--')[1].strip()
    
    return system_prompt, user_prompt

In [66]:
def parse_routine_file(routine_path):
    
    if not os.path.isfile(routine_path):
        print(f"File not found: {routine_path}")
        return []

    with open(routine_path, 'r', encoding="utf-8") as f:
        routine_content = f.read()
    
    routines = []
    current_steps = []
    
    for line in routine_content.splitlines():
        line = line.strip()
        
        if (line == "") and current_steps:
            routines.append(current_steps)
            current_steps = []
        
        else:
            current_steps.append(line)
    
    if current_steps:
        routines.append(current_steps)
        
    return routines

In [67]:
def regenerate_invalid_steps(activity_block, invalid_line, env_data, env_number, prompt_dir, client):
    
    # regenerated string
    result = ""
    
    # --- Extract location (room) from the end of the line: "(bedroom)" ---
    room_match = re.search(r"\(([^()]+)\)\s*$", invalid_line)
    location = room_match.group(1).strip().lower() if room_match else None
    
    # --- Extract time "(07:12 - 07:15)" from the line ---
    time_match = re.search(r"\((\d{1,2}:\d{2} - \d{1,2}:\d{2})\)", invalid_line)
    time_str = time_match.group(1) if time_match else "unknown"
    
    context = "\n".join(activity_block)
    
    # prompt
    system_prompt, user_prompt = load_prompts_from_file(prompt_dir) 
    
    user_prompt_start = (
        f"Invalid Action Line: {invalid_line}\n"
        f"Full Activity Block for Context:\n{context}\n\n"
        f"Object Allowed to Use: {env_data[location]}\n\n"
    )
    
    # regenerate
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_start + user_prompt}
        ]
    )
    
    line = response.choices[0].message.content
    
    # Similarity Check
    langchain_actions = LangChainActions(env_number)
    valid_rooms = {"bathroom", "bedroom", "kitchen", "livingroom"}
    
    # Match action
    action_match = re.search(r"\[(.*?)\]", line)
    action = action_match.group(1) if action_match else None
    action_key = langchain_actions.get_action(action) if action else None
    if action_key is None:
        result += "[]"
        return result
    result += f"[{action_key}] "

    
    # Match object
    objects = re.findall(r"<(.*?)>", line)
    for obj in objects:
        obj = obj.lower().replace("_", "").replace(" ", "")

        if obj in valid_rooms:
            result += f"<{obj}> "
            continue

        object_key = langchain_actions.get_object(obj, filter=lambda metadata: location in metadata['rooms'])
        if object_key is None:
            result += "<> "
            continue
        result += f"<{object_key}> "
    result += f"({time_str}) "
    result += f"({location})"
    
    return result

In [68]:
# Compare the missing objects
def detect_missing_actions(json_file, env_number, parsed_file, grounded_file, output_dir, prompt_dir, client):
    
    parsed_routines = parse_routine_file(parsed_file)
    grounded_routines = parse_routine_file(grounded_file)
    
    total_lines = 0
    missing_amount = 0
    missing_lines = []
      
    ACTION_FORMATS = {
        "walk": 1, "run": 1, "walktowards": 1, "walkforward": 0,
        "turnleft": 0, "turnright": 0, "sit": 1, "standup": 0,
        "grab": 1, "open": 1, "close": 1, "put": 2, "putin": 2,
        "switchon": 1, "switchoff": 1, "drink": 1, "touch": 1, "lookat": 1
    }
    
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    env_key = f"env_{env_number}"
    if env_key not in data:
        raise ValueError(f"Environment {env_number} not found in JSON file.")
    
    env_data = data[env_key]
    
    for i, action_block in enumerate(grounded_routines):
        for j, line in enumerate(action_block):
            total_lines += 1
            
            objects = re.findall(r"<(.*?)>", line)
            expected_objects = len(objects)
            
            if (line != "[]"):
                action_match = re.match(r"\[(.*?)\]", line)
                action = action_match.group(1) if action_match else None
                if action in ACTION_FORMATS:
                    expected_objects = ACTION_FORMATS[action]
            
            # add action detection later
            if (line == "[]") or ("<>" in line) or (len(objects) != expected_objects):  # Detect missing components
                missing_amount += 1   
                count = 0
                
                updated_line = regenerate_invalid_steps(parsed_routines[i], parsed_routines[i][j], env_data, env_number, prompt_dir, client)
                
                while count < 10:
                    objects = re.findall(r"<(.*?)>", updated_line)
                    expected_objects = len(objects)
                    
                    if (updated_line != "[]"):
                        action_match = re.match(r"\[(.*?)\]", updated_line)
                        action = action_match.group(1) if action_match else None
                        if action in ACTION_FORMATS:
                            expected_objects = ACTION_FORMATS[action]                    
                    
                    if (updated_line == "[]") or ("<>" in updated_line) or (len(objects) != expected_objects):
                        updated_line = regenerate_invalid_steps(parsed_routines[i], parsed_routines[i][j], env_data, env_number, prompt_dir, client)
                        count += 1
                    else:
                        break

                missing_lines.append((i + 1, parsed_routines[i][j], line, updated_line))
                print((total_lines, parsed_routines[i][j], line, updated_line))
                grounded_routines[i][j] = updated_line    
                   
    
    if missing_lines:
        print("🔴 Missing Actions and Corresponding Pre-Processed Actions:")
        print(missing_lines)

    base_name = os.path.splitext(os.path.basename(grounded_file))[0]
    out_filename = f"regenerated_{base_name}.txt"
    out_path = os.path.join(output_dir, out_filename)

    with open(out_path, "w", encoding="utf-8") as out_file:
        for block in grounded_routines:
            for updated_line in block:
                out_file.write(updated_line + "\n")
            # Optional: write a blank line to separate blocks
            out_file.write("\n")


In [69]:
def process_one_pair(json_file, parsed_file, grounded_file, output_folder, prompt_dir, client):

    # Make sure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Validate inputs
    if not os.path.exists(parsed_file):
        raise FileNotFoundError(f"Parsed routine file not found: {parsed_file}")

    if not os.path.exists(grounded_file):
        raise FileNotFoundError(f"Grounded routine file not found: {grounded_file}")

    # Extract env ID from filename (expects pattern like 'env_0')
    basename = os.path.basename(grounded_file)
    env_match = re.search(r'env_(\d+)', basename)

    if not env_match:
        raise ValueError(
            "Could not find environment ID in grounded file name. "
            "Expected pattern like 'env_0'."
        )

    env_number = int(env_match.group(1))
    
    print(
        f"\n[AgentSense] Regenerating unresolved actions\n"
        f"[AgentSense] Parsed routine   : {parsed_file}\n"
        f"[AgentSense] Grounded routine : {grounded_file}\n"
        f"[AgentSense] Environment      : env_{env_number}"
    )

    detect_missing_actions(
        json_file=json_file,
        env_number=env_number,
        parsed_file=parsed_file,
        grounded_file=grounded_file,
        output_dir=output_folder,
        prompt_dir=prompt_dir,
        client=client
    )


In [70]:
# =========================
# Configuration (User-editable)
# =========================

# Directory containing prompts for Step 7 regeneration
# These prompts are used to selectively regenerate unresolved actions
# while preserving the original routine structure and timing.
regeneration_prompt_dir = "prompts/regeneration_prompt.txt"

# Parsed routine file (from Step 5)
# This file contains the cleaned, block-level routine BEFORE grounding.
parsed_file = "data/step_5_data/Sarah_routine_env_0_Sunday_parsed.txt"

# Grounded routine file (from Step 6)
# This file contains the SAME routine AFTER action/object grounding,
# where actions and objects are mapped to valid VirtualHome primitives.
grounded_file = "data/step_6_data/Sarah_routine_env_0_Sunday_parsed_grounded.txt"

# IMPORTANT:
# The parsed file and grounded file MUST correspond to the same
# synthetic character, environment, and day.
# Step 7 compares these two versions to identify missing or unresolved
# actions introduced during grounding and selectively regenerates them.
# -----------------------------------------

# Output directory for regenerated result
output_folder = "data/step_7_data"

process_one_pair(
    json_file="room_objects.json",
    parsed_file=parsed_file,
    grounded_file=grounded_file,
    output_folder=output_folder,
    prompt_dir=regeneration_prompt_dir,
    client=client
);



[AgentSense] Regenerating unresolved actions
[AgentSense] Parsed routine   : data/step_5_data/Sarah_routine_env_0_Sunday_parsed.txt
[AgentSense] Grounded routine : data/step_6_data/Sarah_routine_env_0_Sunday_parsed_grounded.txt
[AgentSense] Environment      : env_0
17 17
(20, '[turnon] <faucet> (08:30 - 08:31) (bathroom)', '[]', '[switchon] <faucet> (08:30 - 08:31) (bathroom)')
(36, '[grab] <chail> (08:47 - 08:48) (bedroom)', '[grab] <> (08:47 - 08:48) (bedroom)', '[grab] <chair> (08:47 - 08:48) (bedroom)')
(313, '[wait] <tv> (19:55 - 20:12) (livingroom)', '[]', '[]')
(324, '[eat] <plum> (20:31 - 20:32) (livingroom)', '[]', '[grab] <plum> (20:31 - 20:32) (livingroom)')
(341, '[walk] <sink> (21:14 - 21:15) (bathroom)', '[walk] <> (21:14 - 21:15) (bathroom)', '[walk] <bathroom> (21:14 - 21:15) (bathroom)')
(378, '[sit] <floor> (22:22 - 22:24) (bedroom)', '[sit] <> (22:22 - 22:24) (bedroom)', '[sit] <rug> (22:22 - 22:24) (bedroom)')
(387, '[turnon] <tablelamp> (22:32 - 22:32) (bedroom)',